# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the FAIR² Croissant metadata and records using `mlcroissant`. The Croissant schema URL defines the structure and content of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the schema URL (as provided)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their respective `@id`s. All components are referenced by their `@id` for rigor and reproducibility per the Croissant specification.

*Tip: If the dataset has only one main recordset, its `@id` will be used throughout the notebook. Otherwise, iterate through all recordsets and their fields by their `@id`s.*

In [ ]:
# List all available record sets in the dataset, using @id
record_sets = list(dataset.record_sets)
if not record_sets:
    raise ValueError("No record sets found in dataset. Please check the schema.")
print("Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}")

example_record_set_id = record_sets[0]['@id'] if record_sets else None

# List fields with their @id and data type for each record set
print(f"\nFields for record set '{example_record_set_id}':")
fields = dataset.record_set(example_record_set_id).fields
for field in fields:
    print(f"- {field['@id']} (dataType: {field.get('dataType', 'unknown')})")

## 3. Data Extraction

Load data from the primary record set (using its `@id`) into a DataFrame for further analysis. We will also display the DataFrame's columns and the first few rows to preview the data.

In [ ]:
# For this notebook, use the first record set (if multiple exist)
selected_record_set_id = example_record_set_id
print(f"Extracting records from record set: {selected_record_set_id}")

records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print(f"Columns in DataFrame: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's perform common EDA tasks: filter on a numeric field, normalize it, and group by a categorical column. All fields are referenced by their `@id`.

> **Note:** Replace `numeric_field_id` and `group_field_id` with actual `@id` values identified in the overview.

In [ ]:
# Example: Use '@id' fields for the operations.
# Choose a representative numeric field (e.g., 'cr:Age' or similar; replace with one listed above)

# List all columns for easier selection if unsure
print("All columns (field @id's):", df.columns.tolist())

# Example selection (update these if a different field is desired):
numeric_field_id = df.select_dtypes(include='number').columns[0]
# Choose a group field that is categorical
candidate_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
group_field_id = candidate_group_fields[0] if candidate_group_fields else None

if numeric_field_id:
    # Filtering on an arbitrary threshold (e.g., > median)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} found.")
    print(filtered_df.head())

    # Normalize numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id, if found
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found in record set for EDA.")

## 5. Visualization

Visualize the distribution of the numeric field and the group-wise means. If you know clinical variables of interest, adapt the visualizations accordingly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if numeric_field_id is available
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Group violin/boxplot visualization
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- This notebook demonstrated data loading, exploration, EDA, and visualization using the `mlcroissant` library and FAIR² dataset schema.
- All references to data entities, record sets, and fields were made using their `@id` values for precise, schema-driven analysis.
- The approach is flexible: you can change field selection by updating the relevant `@id` variables or function arguments.

Further steps could include advanced modeling, subsetting by clinical variables, or exporting cleaned subsets for domain-specific analysis.